# Task-01 & Task-05: VisDrone Dataset EDA + Results Visualization
**Antlings Internship – AI/ML Drone Human Detection & Counting System**

In [ ]:
import os, cv2, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from collections import Counter

# ── Config ────────────────────────────────────────────────────────────────────
VISDRONE_ROOT = '../VisDrone/VisDrone2019-DET-train'
YOLO_DATASET  = '../dataset_yolo'

CLASS_NAMES = {
    0: 'ignored',    1: 'pedestrian', 2: 'people',
    3: 'bicycle',    4: 'car',        5: 'van',
    6: 'truck',      7: 'tricycle',   8: 'awning-tricycle',
    9: 'bus',       10: 'motor',     11: 'others'
}

print('Setup done ✅')

## 1. Class Distribution in Training Set

In [ ]:
ann_dir = Path(VISDRONE_ROOT) / 'annotations'
class_counts = Counter()

for ann_file in list(ann_dir.glob('*.txt'))[:500]:   # sample 500 for speed
    for line in ann_file.read_text().strip().split('\n'):
        parts = line.split(',')
        if len(parts) >= 6:
            class_counts[int(parts[5])] += 1

fig, ax = plt.subplots(figsize=(12, 5))
labels = [CLASS_NAMES.get(k, k) for k in sorted(class_counts)]
values = [class_counts[k] for k in sorted(class_counts)]
colors = ['#e74c3c' if k in (1,2) else '#3498db' if k in (4,5,6,9) else '#95a5a6'
          for k in sorted(class_counts)]

ax.bar(labels, values, color=colors)
ax.set_title('Object Class Distribution (sample 500 images)', fontsize=14)
ax.set_xlabel('Class')
ax.set_ylabel('Count')
plt.xticks(rotation=30, ha='right')

patches = [
    mpatches.Patch(color='#e74c3c', label='→ person'),
    mpatches.Patch(color='#3498db', label='→ car'),
    mpatches.Patch(color='#95a5a6', label='ignored'),
]
ax.legend(handles=patches)
plt.tight_layout()
plt.savefig('../outputs/class_distribution.png', dpi=150)
plt.show()

## 2. Bounding Box Size Distribution

In [ ]:
widths, heights = [], []
for ann_file in list(ann_dir.glob('*.txt'))[:300]:
    for line in ann_file.read_text().strip().split('\n'):
        parts = line.split(',')
        if len(parts) >= 6 and int(parts[5]) in (1,2,4,5,6,9):
            widths.append(int(parts[2]))
            heights.append(int(parts[3]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths, bins=60, color='#2ecc71', alpha=0.8, edgecolor='white')
axes[0].set_title('Bounding Box Width Distribution')
axes[0].set_xlabel('Width (px)')
axes[0].axvline(32, color='red', linestyle='--', label='32px threshold')
axes[0].legend()

axes[1].hist(heights, bins=60, color='#e67e22', alpha=0.8, edgecolor='white')
axes[1].set_title('Bounding Box Height Distribution')
axes[1].set_xlabel('Height (px)')
axes[1].axvline(32, color='red', linestyle='--', label='32px threshold')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/bbox_distribution.png', dpi=150)
plt.show()
print(f'Median box: {np.median(widths):.1f}w × {np.median(heights):.1f}h px')
small = sum(1 for w, h in zip(widths, heights) if w < 32 and h < 32)
print(f'Small objects (<32px both dims): {100*small/len(widths):.1f}%')

## 3. Sample Images with GT Annotations

In [ ]:
img_dir = Path(VISDRONE_ROOT) / 'images'
img_paths = random.sample(list(img_dir.glob('*.jpg')), 4)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, img_path in zip(axes.flat, img_paths):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    ann  = Path(VISDRONE_ROOT) / 'annotations' / (img_path.stem + '.txt')
    for line in ann.read_text().strip().split('\n'):
        p = line.split(',')
        if len(p) < 6: continue
        bx,by,bw,bh,_,cat = int(p[0]),int(p[1]),int(p[2]),int(p[3]),p[4],int(p[5])
        if cat in (1,2):
            color = (46, 204, 113)
        elif cat in (4,5,6,9):
            color = (52, 152, 219)
        else:
            continue
        import matplotlib.patches as mp
        ax.add_patch(mp.Rectangle((bx,by), bw, bh,
                                   linewidth=1, edgecolor=np.array(color)/255, fill=False))
    ax.imshow(img)
    ax.set_title(img_path.name, fontsize=9)
    ax.axis('off')

plt.suptitle('Sample VisDrone Training Images with GT Boxes\n🟢 person  🔵 car', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/sample_gt.png', dpi=150)
plt.show()

## 4. Training Metrics (after training)

In [ ]:
results_csv = '../runs/train/visdrone_exp/results.csv'
if Path(results_csv).exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    metrics_to_plot = [
        ('train/box_loss', 'Box Loss (train)', '#e74c3c'),
        ('train/cls_loss', 'Class Loss (train)', '#e67e22'),
        ('metrics/mAP50(B)', 'mAP@0.5', '#2ecc71'),
        ('metrics/mAP50-95(B)', 'mAP@0.5:0.95', '#3498db'),
        ('metrics/precision(B)', 'Precision', '#9b59b6'),
        ('metrics/recall(B)', 'Recall', '#1abc9c'),
    ]
    for ax, (col, title, color) in zip(axes.flat, metrics_to_plot):
        if col in df.columns:
            ax.plot(df['epoch'], df[col], color=color, linewidth=2)
            ax.set_title(title, fontsize=11)
            ax.set_xlabel('Epoch')
            ax.grid(alpha=0.3)
    plt.suptitle('YOLOv8 Training Curves', fontsize=14)
    plt.tight_layout()
    plt.savefig('../outputs/training_curves.png', dpi=150)
    plt.show()
else:
    print('Train first: python src/train.py --data dataset_yolo/dataset.yaml')